# 04 - Model Training & Evaluation

Train Linear Regression, Random Forest, XGBoost with spatial cross-validation. Feature importance and failure analysis.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from src.data_loader import load_features
from src.models import (
    train_linear_regression,
    train_random_forest,
    train_xgboost,
    get_feature_importance
)
from src.evaluation import (
    spatial_cross_validate,
    compute_metrics,
    paired_ttest,
    identify_failure_cases,
    plot_predicted_vs_actual,
    plot_feature_importance
)
from src.utils import get_data_dir, get_experiments_dir
import joblib

try:
    X_df, y = load_features()
except FileNotFoundError:
    from src.features import build_feature_matrix
    from src.synthetic_data import generate_synthetic_dataset
    df = generate_synthetic_dataset()
    X_df, y = build_feature_matrix(df)

feature_cols = [c for c in X_df.columns if c != 'field_id']
X = X_df[feature_cols].fillna(0).values
y = y.values
le = LabelEncoder()
groups = le.fit_transform(X_df['field_id'])

## 1. Spatial Cross-Validation

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

results = {}

# Linear Regression
m_lr, _ = train_linear_regression(X, y)
cv_lr, yt_lr, yp_lr = spatial_cross_validate(X, y, groups, lambda: LinearRegression(), n_splits=5)
results['Linear Regression'] = cv_lr
print('Linear Regression:', {k: round(v, 4) for k, v in cv_lr.items() if k != 'fold_metrics'})

# Random Forest
m_rf, _ = train_random_forest(X, y, groups=groups, cv=5)
yp_rf = np.zeros_like(y)
from sklearn.model_selection import GroupKFold
for tr, val in GroupKFold(5).split(X, y, groups=groups):
    m_rf.fit(X[tr], y[tr])
    yp_rf[val] = m_rf.predict(X[val])
results['Random Forest'] = compute_metrics(y, yp_rf)
print('Random Forest:', results['Random Forest'])

# XGBoost
m_xgb, _ = train_xgboost(X, y, groups=groups, cv=5)
yp_xgb = np.zeros_like(y)
for tr, val in GroupKFold(5).split(X, y, groups=groups):
    m_xgb.fit(X[tr], y[tr])
    yp_xgb[val] = m_xgb.predict(X[val])
results['XGBoost'] = compute_metrics(y, yp_xgb)
print('XGBoost:', results['XGBoost'])

## 2. Statistical Comparison (Paired t-test)

In [ ]:
t_stat, p_val = paired_ttest(y, yp_rf, yp_xgb)
print(f'Paired t-test RF vs XGBoost: t={t_stat:.4f}, p={p_val:.4f}')
print('XGBoost significantly better' if p_val < 0.05 else 'No significant difference')

## 3. Predicted vs Actual & Feature Importance

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_predicted_vs_actual(y, yp_xgb, ax=axes[0], title='XGBoost')
imp_df = get_feature_importance(m_xgb, feature_cols)
plot_feature_importance(imp_df, ax=axes[1], top_n=12)
plt.tight_layout()
plt.savefig(get_experiments_dir() / 'results' / 'predictions_and_importance.png', dpi=150)
plt.show()

## 4. Failure Analysis

In [ ]:
failure_df = identify_failure_cases(X_df, y, yp_xgb, X_df['field_id'].values, n_worst=5)
print('Worst prediction errors:')
print(failure_df)

In [ ]:
# Save best model
get_experiments_dir().mkdir(parents=True, exist_ok=True)
(get_experiments_dir() / 'results').mkdir(exist_ok=True)
joblib.dump(m_xgb, get_experiments_dir() / 'results' / 'best_model_xgb.joblib')
print('Saved best model to experiments/results/best_model_xgb.joblib')